In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import holidays
import json
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error

df_visitas = pd.read_csv('../data/dataset_global_raw.csv')
df_visitas['fecha'] = pd.to_datetime(df_visitas['fecha'])

df_clima = pd.read_csv('../data/raw/clima_cache.csv')
df_clima['fecha'] = pd.to_datetime(df_clima['fecha'])

with open('../todas_las_ubicaciones.json', 'r', encoding='utf-8') as f:
    datos_loc = json.load(f)

mapa_regiones = {}
mapa_nombres = {}
for org in datos_loc:
    for loc in org.get('locations', []):
        mapa_nombres[loc['uuid']] = loc['name']
        if 'region_code' in loc:
            mapa_regiones[loc['uuid']] = loc['region_code']

df_visitas['region_code'] = df_visitas['location_id'].map(mapa_regiones)
df_visitas['nombre_tienda'] = df_visitas['location_id'].map(mapa_nombres)

df_master = pd.merge(
    df_visitas, df_clima, 
    left_on=['fecha', 'location_id'], right_on=['fecha', 'location_uuid'], how='left'
)
df_master['llueve'] = df_master['llueve'].fillna(0).astype(int)

def asignar_festivo(row):
    try:
        if pd.isna(row['region_code']): return 0
        calendario = holidays.Spain(subdiv=row['region_code'], years=row['fecha'].year)
        return 1 if row['fecha'] in calendario else 0
    except:
        return 0

df_master['es_festivo'] = df_master.apply(asignar_festivo, axis=1)
df_master['es_finde'] = df_master['fecha'].dt.dayofweek.isin([5, 6]).astype(int)

df_master['dia_semana'] = df_master['fecha'].dt.dayofweek
df_master['dia_mes'] = df_master['fecha'].dt.day
df_master['mes'] = df_master['fecha'].dt.month

In [2]:
uuid_objetivo = '251e7f40-95c7-4678-aa48-df1b90e3461c'
nombre_objetivo = mapa_nombres.get(uuid_objetivo, "Ubicacion Desconocida")

print("="*60)
print(f"INICIANDO LABORATORIO PARA: {nombre_objetivo}")
print(f"UUID: {uuid_objetivo}")
print("="*60)

df_tienda = df_master[df_master['location_id'] == uuid_objetivo].copy()
df_tienda = df_tienda.groupby('fecha').agg({
    'total_visits': 'sum',
    'llueve': 'max',
    'es_festivo': 'max',
    'es_finde': 'max',
    'dia_semana': 'max',
    'dia_mes': 'max',
    'mes': 'max'
}).reset_index().sort_values('fecha').reset_index(drop=True)

df_tienda['lag_1d'] = df_tienda['total_visits'].shift(1)
df_tienda['lag_7d'] = df_tienda['total_visits'].shift(7)
df_tienda['media_7d'] = df_tienda['total_visits'].rolling(7).mean()

df_tienda = df_tienda.dropna().reset_index(drop=True)

print(f"\nTotal de dias validos para entrenar: {len(df_tienda)} dias")
print("\nVISTAZO AL TENSOR DE DATOS (Primeras 5 filas):")
display(df_tienda.head())

INICIANDO LABORATORIO PARA: Madrid Gran Via
UUID: 251e7f40-95c7-4678-aa48-df1b90e3461c

Total de dias validos para entrenar: 217 dias

VISTAZO AL TENSOR DE DATOS (Primeras 5 filas):


,fecha,total_visits,llueve,es_festivo,es_finde,dia_semana,dia_mes,mes,lag_1d,lag_7d,media_7d
0,2025-09-09,29079,0,0,0,1,9,9,29150.0,32339.0,37419.000000
1,2025-09-10,34488,1,0,0,2,10,9,29079.0,35974.0,37206.714286
2,2025-09-11,34787,0,0,0,3,11,9,34488.0,35243.0,37141.571429
3,2025-09-12,40510,0,0,0,4,12,9,34787.0,44717.0,36540.571429
4,2025-09-13,43400,0,0,1,5,13,9,40510.0,42270.0,36702.000000


In [3]:

fecha_corte_test = '2026-04-05' 

train = df_tienda[df_tienda['fecha'] < fecha_corte_test].copy()
test = df_tienda[df_tienda['fecha'] >= fecha_corte_test].copy()

print("="*60)
print("CONFIGURACION DEL PERIODO DE ANALISIS")
print("="*60)
print(f"ENTRENAMIENTO: Desde {train['fecha'].min().date()} hasta {train['fecha'].max().date()} ({len(train)} dias)")
print(f"TEST (VALIDACION): Desde {test['fecha'].min().date()} hasta {test['fecha'].max().date()} ({len(test)} dias)")
print("="*60)

features = ['es_finde', 'es_festivo', 'llueve', 'dia_semana', 'dia_mes', 'mes', 'lag_1d', 'lag_7d', 'media_7d']

X_train, y_train = train[features], train['total_visits']
X_test, y_test = test[features], test['total_visits']


modelo = xgb.XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
modelo.fit(X_train, y_train)

test['prediccion'] = modelo.predict(X_test)
test['prediccion'] = test['prediccion'].apply(lambda x: max(0, round(x)))

mae = mean_absolute_error(test['total_visits'], test['prediccion'])
suma_errores = np.sum(np.abs(test['total_visits'] - test['prediccion']))
suma_real = np.sum(test['total_visits'])
wmape = suma_errores / suma_real if suma_real > 0 else 0
accuracy = (1 - wmape) * 100

print(f"RESULTADOS EN EL PERIODO ELEGIDO:")
print(f"MAE: {mae:.1f} visitas | Accuracy: {accuracy:.2f}%")

CONFIGURACION DEL PERIODO DE ANALISIS
ENTRENAMIENTO: Desde 2025-09-09 hasta 2026-04-04 (208 dias)
TEST (VALIDACION): Desde 2026-04-05 hasta 2026-04-13 (9 dias)
RESULTADOS EN EL PERIODO ELEGIDO:
MAE: 6451.7 visitas | Accuracy: 87.01%


In [4]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=test['fecha'], 
    y=test['total_visits'],
    mode='lines+markers',
    name='Realidad (Visitantes)',
    line=dict(color='black', width=2),
    marker=dict(size=6)
))

fig.add_trace(go.Scatter(
    x=test['fecha'], 
    y=test['prediccion'],
    mode='lines+markers',
    name='Modelo XGBoost (Prediccion)',
    line=dict(color='#8e44ad', width=2, dash='dot'),
    marker=dict(size=6, symbol='cross')
))

for index, row in test.iterrows():
    fig.add_trace(go.Scatter(
        x=[row['fecha'], row['fecha']],
        y=[row['total_visits'], row['prediccion']],
        mode='lines',
        line=dict(color='red', width=1),
        showlegend=False,
        hoverinfo='skip'
    ))

fig.update_layout(
    title=f'Inferencia vs Realidad en {nombre_objetivo} (Test Set)',
    xaxis_title='Fecha',
    yaxis_title='Visitantes Totales',
    hovermode='x unified',
    template='plotly_white',
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()